# 🤖 Week 2 — PydanticAI Core
> **Goal:** Build real AI agents that return typed Python objects — not messy strings.

---
### What you will learn
1. PydanticAI Agent basics
2. System prompts vs user prompts
3. Structured (typed) outputs with `result_type`
4. Synchronous & async agent runs
5. Mini-project: **AI Resume Analyzer** returning structured JSON

## 1 — Install & Setup

In [61]:
%pip install pydantic-ai python-dotenv groq rich --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\Nidhi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [62]:
import os
from dotenv import load_dotenv
load_dotenv()  # loads .env file from project root

# Make sure your GROQ_API_KEY is set
assert os.getenv("GROQ_API_KEY"), "Set GROQ_API_KEY in .env!"
print("Environment loaded successfully.")

Environment loaded successfully.


## 2 — Your first PydanticAI Agent

In [63]:
## 2 — Your first PydanticAI Agent
from pydantic_ai import Agent
agent = Agent(
    model="groq:llama-3.1-8b-instant",
    system_prompt = "You are a helpful assistant. keep answer under 3 sentence"
)

result = await agent.run("what is Python")
print(result.output)

print(f"\n usage:{result.usage()}")


Python is a high-level, interpreted programming language known for its simplicity, readability, and flexibility. It is widely used for various purposes such as web development, data analysis, artificial intelligence, and scripting, making it a popular choice among developers and data scientists. Python's syntax is easy to learn and understand, allowing both beginners and experts to create efficient and effective code.

 usage:RunUsage(input_tokens=50, output_tokens=74, requests=1)


## 3 — Typed output: the killer feature

In [64]:
## 3 — Typed output: the killer feature
from pydantic import BaseModel, Field
from typing import List
from pydantic_ai import Agent

class CountryFacts(BaseModel):
    country: str
    capital: str
    population_millions: float
    languages: List[str]
    fun_fact: str


facts_agent = Agent(
    model="groq:llama-3.1-8b-instant",
    output_type=CountryFacts,
    system_prompt="Return ONLY valid JSON with country facts.",
)

result = await facts_agent.run("Tell me about Japan")

facts = result.output


print("Country :", facts.country)
print("Capital :", facts.capital)
print("Population:", facts.population_millions, "M")
print("Languages:", ", ".join(facts.languages))
print("Fun fact:", facts.fun_fact)

Country : Japan
Capital : Tokyo
Population: 128.0 M
Languages: Japanese
Fun fact: Japan is home to the world's oldest known restaurant, Honke Owariya, which has been serving traditional Japanese dishes since 1465.


## 4 — Dynamic system prompts

In [65]:

## 4 — Dynamic system prompts
from pydantic_ai import Agent, RunContext
from dataclasses import dataclass

@dataclass
class UserContext:
    username: str
    expertise_level: str

agent = Agent(
    model="groq:llama-3.1-8b-instant",
    deps_type=UserContext,
)

@agent.system_prompt
def prompt(ctx: RunContext[UserContext]) -> str:
    lvl = ctx.deps.expertise_level
    # ctx = current run data
    # deps = user data we passed
    # extracting users level here
    return f"Tutor for {ctx.deps.username}. {'Simple explanation.' if lvl=='beginner' else 'Technical explanation.'}"
    # if user is begginer -> simple explaination
    # else -> technical explaination

# run
for level in ["beginner", "expert"]:
    res = await agent.run(
        "What is a list comprehension?",
        deps=UserContext("Nidhi", level)
    )
    print(f"\n=== {level} ===")
    print(res.output)


=== beginner ===
**List Comprehension: A Simple Explanation**

A list comprehension is a powerful tool in Python that allows you to create new lists from existing ones in a concise and readable way. It's a shorthand for a loop that creates a new list.

**Basic Syntax:**
```python
new_list = [expression for element in iterable]
```
Let's break it down:

* `expression`: This is what you want to do with each element in the `iterable`. It can be a simple operation, like `element * 2`, or a more complex expression.
* `element`: This is the variable that represents each element in the `iterable`.
* `iterable`: This is the list or any other iterable (like a tuple, set, or dictionary) that you want to create a new list from.

**Example:**
```python
numbers = [1, 2, 3, 4, 5]
squared_numbers = [num ** 2 for num in numbers]
print(squared_numbers)  # Output: [1, 4, 9, 16, 25]
```
In this example, we create a new list `squared_numbers` by squaring each number in the original list `numbers`. The li

## 5 — Multi-turn conversation (message history)

In [66]:
## 5 — Multi-turn conversation (message history)
from pydantic_ai import Agent

chat_agent = Agent(
    model="groq:llama-3.1-8b-instant",
    system_prompt="You are a friendly Python tutor. Remember previous messages.",
)

# Turn 1
r1 = await chat_agent.run("My name is Nidhi and I am learning Python.")
print("Turn 1:", r1.output)

# Turn 2 (with memory)
r2 = await chat_agent.run(
    "What is my name and what am I learning?",
    message_history=r1.new_messages(),
)
print("\nTurn 2:", r2.output)

Turn 1: Hi Nidhi, it's great to meet you. I'd be happy to help you learn Python. What level are you at in your Python learning journey? Are you a complete beginner or do you have some experience already?

Turn 2: Your name is Nidhi, and you're learning Python!


## 6 — Async agents (for production apps)

In [67]:
from pydantic_ai import Agent
from pydantic import BaseModel
from typing import List

class KeyPoints(BaseModel):
    topic: str
    points: List[str]

agent = Agent(
    model="groq:llama-3.1-8b-instant",
    output_type=KeyPoints,
    # THIS is the correct retry control (not model_settings)
    retries=5,
    system_prompt = (

        "You are a STRICT JSON generator.\n"
        "DO NOT use tools or search.\n"
        "DO NOT use function calls.\n"
        "Return ONLY valid JSON matching schema."
          ),
        )

try:
    res = await agent.run("FastAPI for building REST APIs")

    kp = res.output

    print("Topic:", kp.topic)

    for i, p in enumerate(kp.points, 1):
        print(i, p)

except Exception as e:
    print("Failed safely:", e)
 


Topic: FastAPI
1 FastAPI is a modern, fast high-performance web framework for building APIs.
2 FastAPI uses standard Python type hints for routing, validation, and serialization/deserialization, making it a great choice for building web APIs.


## 🏆 Mini-Project: AI Resume Analyzer
Build an agent that reads a resume text and returns structured analysis.

In [68]:
# ## 🏆 Mini-Project: AI Resume Analyzer
from pydantic import BaseModel, Field
from typing import List
from pydantic_ai import Agent

class Resume(BaseModel):
    name: str
    exp_years: int
    skills: List[str]
    education: str
    strengths: List[str]
    gaps: List[str]
    score: int = Field(ge=1, le=10)   
 
agent = Agent(
    model="groq:llama-3.1-8b-instant",
    output_type=Resume,
    # THIS is the correct retry control (not model_settings)
    system_prompt = (
        "Analyze resume carefully. "
        "Score MUST be between 1 and 10. "
        "Return valid structured JSON only."

    ),
    )
  
sample = """
Name: Sarah Chen
Experience: 4 years Python dev
Skills: Python, FastAPI, Docker, AWS
Education: BSc Computer Science
"""
res = await agent.run(f"Analyze this resume :\n{sample}")
data = res.output


print("Name:", data.name)
print("Experience:", data.exp_years)
print("Skills:", data.skills)
print("Score:", data.score)

Name: Sarah Chen
Experience: 4
Skills: ['Python', 'FastAPI', 'Docker', 'AWS']
Score: 8


## Week 2 — Summary
| Concept | How |
|---|---|
| Create an agent | `Agent(model=..., system_prompt=...)` |
| Typed output | `result_type=YourModel` |
| Dynamic system prompt | `@agent.system_prompt` decorator |
| User context (deps) | `deps_type=YourDataclass` |
| Multi-turn chat | `message_history=result.new_messages()` |
| Async agents | `await agent.run(...)` |

**Next: Week 3 — Groq Integration (speed)**